# Lab 2 appendix — where does a table actually live?*Not marked, and not part of the rubric.* Everything in `lab.ipynb` was about how Spark **runs**a query. This is about where the table it reads from is **kept**, because the two are answered bycompletely different machinery.It explains the `metastore_db/` and `spark-warehouse/` directories that appear in your repo root,and it is the local counterpart to Unity Catalog.**You do not need to have run `lab.ipynb` first** — the setup cells below build the `trips` tableif it is not already there.To clean up everything this notebook creates:```shrm -rf custom_warehouse taxi_catalog_db```

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def find_repo_root() -> Path:
    """The directory holding get_data.py, searching upward from the working directory.

    A kernel starts in the notebook's own directory (labs/L2/), so walk up rather than
    assuming anything about where you launched Jupyter from.
    """
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "get_data.py").exists():
            return candidate
    raise RuntimeError("Could not find the repo root — open this notebook from inside the repo.")


# Everything below assumes the repo root as the working directory: the dataset path, and the
# spark-warehouse/ and metastore_db/ directories Spark creates next to it. Move there once,
# here, before Spark starts.
os.chdir(find_repo_root())
print(f"working directory: {Path.cwd()}")

# A kernel started from an IDE never sources your shell profile, so JAVA_HOME can be missing
# here even though `java -version` works fine in a terminal. Resolve it before PySpark loads.
if "JAVA_HOME" not in os.environ:
    if sys.platform == "darwin":
        os.environ["JAVA_HOME"] = subprocess.check_output(["brew", "--prefix", "openjdk@17"], text=True).strip()
    else:
        raise RuntimeError("Set JAVA_HOME for your user account, then restart the kernel.")
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ.get("PATH", "")
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)

# Imported after the block above on purpose: PySpark looks for the JVM at import time.
from pyspark.sql import SparkSession  # noqa: E402
from pyspark.sql import functions as F  # noqa: E402

In [ ]:
# local[4] rather than local[*]: this lab measures parallelism, so the number of slots has
# to be a number you know rather than "however many cores this laptop has".
CORES = 4

spark = (
    SparkSession.builder.appName("L2-execution")
    .master(f"local[{CORES}]")
    # A persistent catalog (a local Derby metastore under metastore_db/). Without it Spark
    # forgets its tables when the kernel dies but leaves their files in spark-warehouse/,
    # and the *second* run of this notebook fails with LOCATION_ALREADY_EXISTS.
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark {spark.version}, master {spark.sparkContext.master}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

## Setup — a session and a `trips` tableSame session settings as the lab. The table is created only if the lab has not already made it.

In [ ]:
SRC = "data/yellow_tripdata_2023-01.parquet"
if not Path(SRC).exists():
    raise SystemExit(f"Missing {SRC}\nRun: uv run python get_data.py --dataset taxi")

spark.sparkContext.setJobDescription("Appendix: build the trips table")

trips = spark.read.parquet(SRC).select(
    F.col("tpep_pickup_datetime").cast("timestamp").alias("pickup_ts"),
    F.col("passenger_count").cast("int").alias("passenger_count"),
    F.col("trip_distance").cast("double").alias("trip_distance"),
    F.col("fare_amount").cast("double").alias("fare_amount"),
    F.col("tip_amount").cast("double").alias("tip_amount"),
    F.col("payment_type").cast("int").alias("payment_type"),
    F.col("PULocationID").cast("int").alias("pu_location_id"),
)

# Reuse the lab's table when it exists — writing 3 million rows again just to read them back
# would be pure waste, and this notebook is meant to stand on its own either way.
if spark.catalog.tableExists("spark_catalog.default.trips"):
    print("trips table already exists — reusing it")
else:
    trips.write.mode("overwrite").saveAsTable("trips")
    print("trips table created")

## Appendix — where does a table actually live?

*Not marked. Everything above was about how Spark **runs** a query; this is about where the
table it read from is **kept**, because the two are answered by completely different machinery.*

Every table in Spark has a three-part name:

```
catalog . database . table
```

- **catalog** — a pluggable source of metadata. You have been using `spark_catalog`, the
  built-in one, without naming it. It is the only catalog Spark ships enabled.
- **database** — a namespace inside a catalog, also called a *schema*. Yours is `default`.
- **table** — the thing itself.

So the `trips` table you built in the setup cell is really `spark_catalog.default.trips`.
Run the cell below to confirm it.

In [ ]:
spark.sparkContext.setJobDescription("Appendix: inspect the catalog")

print("catalogs :", [c.name for c in spark.catalog.listCatalogs()])
print("current  :", spark.catalog.currentCatalog(), "/", spark.catalog.currentDatabase())
print("databases:", [d.name for d in spark.catalog.listDatabases()])
print()

# The fully qualified name works exactly like the short one.
print("trips row count via the full name:", spark.table("spark_catalog.default.trips").count())
print()

# DESCRIBE EXTENDED is how you ask the catalog what it knows about a table.
for row in spark.sql("DESCRIBE EXTENDED trips").collect():
    if row[0] in ("Catalog", "Database", "Table", "Type", "Provider", "Location"):
        print(f"  {row[0]:<10} {row[1]}")

### Two directories, two different jobs

Note the `Location` printed above: it points inside `spark-warehouse/`. That is the split
worth understanding, because the two directories in your repo root are not variations on the
same idea.

| | `metastore_db/` | `spark-warehouse/` |
|---|---|---|
| Stores | *that* the table exists — its name, columns, format, location | the rows themselves |
| Format | an embedded Apache Derby database (Java's SQLite) | Parquet files |
| Controlled by | `javax.jdo.option.ConnectionURL` | `spark.sql.warehouse.dir` |

Both are defaults belonging to `spark_catalog`, not laws of Spark. The name `metastore_db`
comes from a string compiled into the Hive jar that ships inside PySpark —
`jdbc:derby:;databaseName=metastore_db;create=true` — which is why it appears in whatever
directory the driver happens to start in.

They are two halves of one thing, so **delete them together or not at all**. Remove
`metastore_db/` alone and Spark forgets the table while its files remain, and the next
`saveAsTable` fails with `LOCATION_ALREADY_EXISTS` — a directory occupied by a table nobody
remembers. Remove `spark-warehouse/` alone and you get the mirror image: a catalogue entry
pointing at nothing.

### Storage is per-database, not global

`spark.sql.warehouse.dir` is only the *default* location. A database can be given its own.

In [ ]:
spark.sparkContext.setJobDescription("Appendix: database with a custom location")

CUSTOM_DIR = str(Path.cwd() / "custom_warehouse")

spark.sql(f"CREATE DATABASE IF NOT EXISTS taxi_db LOCATION '{CUSTOM_DIR}'")

# A small summary of the trips data — a couple of hundred rows, not three million.
hourly = trips.groupBy(F.hour("pickup_ts").alias("hour"), "payment_type").agg(
    F.count("*").alias("trip_count"), F.avg("fare_amount").alias("avg_fare")
)
hourly.write.mode("overwrite").saveAsTable("taxi_db.hourly_summary")

location = [r[1] for r in spark.sql("DESCRIBE EXTENDED taxi_db.hourly_summary").collect() if r[0] == "Location"][0]
print("default database  ->", [r[1] for r in spark.sql("DESCRIBE EXTENDED trips").collect() if r[0] == "Location"][0])
print("taxi_db           ->", location)
print()
print("Same catalog, same metastore_db — only the bytes moved.")

### A second catalog

A database moves the *data*. A **catalog** replaces the metadata layer itself — where Spark
looks tables up in the first place. This is the Catalog Plugin API, and it is how Delta Lake,
Iceberg and Unity Catalog attach to Spark.

You can register one without rebuilding the session: `spark.sql.catalog.<name>` names the
implementation class, and `spark.sql.catalog.<name>.*` configures it. Below we use
`JDBCTableCatalog`, which keeps its metadata *and* its data in a JDBC database — pointed at
its own embedded Derby instance, entirely separate from `metastore_db`.

In [ ]:
spark.sparkContext.setJobDescription("Appendix: write into a custom catalog")

# Register a second catalog at runtime — no new SparkSession needed.
spark.conf.set("spark.sql.catalog.taxi_cat", "org.apache.spark.sql.execution.datasources.v2.jdbc.JDBCTableCatalog")
spark.conf.set("spark.sql.catalog.taxi_cat.url", "jdbc:derby:taxi_catalog_db;create=true")
spark.conf.set("spark.sql.catalog.taxi_cat.driver", "org.apache.derby.jdbc.EmbeddedDriver")

spark.sql("CREATE SCHEMA IF NOT EXISTS taxi_cat.curated")

# writeTo() is the DataFrameWriterV2 API. saveAsTable() resolves this name fine, but it
# issues CREATE TABLE ... USING parquet, and a JDBC catalog has no file provider to
# satisfy that — it rejects the command with NOT_SUPPORTED_IN_JDBC_CATALOG.
hourly.writeTo("taxi_cat.curated.hourly_summary").createOrReplace()

print("rows read back:", spark.table("taxi_cat.curated.hourly_summary").count())
print("catalogs now  :", [c.name for c in spark.catalog.listCatalogs()])
print("tables        :", [(r[0], r[1]) for r in spark.sql("SHOW TABLES IN taxi_cat.curated").collect()])
print()

# One query, both catalogs. Spark resolves each name against its own metadata source.
spark.sql("""
    SELECT w.hour, w.trip_count AS from_warehouse, c.trip_count AS from_custom_catalog
    FROM taxi_db.hourly_summary w
    JOIN taxi_cat.curated.hourly_summary c
      ON w.hour = c.hour AND w.payment_type = c.payment_type
    WHERE w.payment_type = 1
    ORDER BY w.hour
""").show(5)

In [ ]:
spark.sparkContext.setJobDescription("Appendix: what is on disk")

for name in ["metastore_db", "spark-warehouse", "custom_warehouse", "taxi_catalog_db"]:
    path = Path(name)
    if not path.exists():
        print(f"{name:<18} (absent)")
        continue
    parquet = len(list(path.rglob("*.parquet")))
    print(
        f"{name:<18} {sum(f.stat().st_size for f in path.rglob('*') if f.is_file()) / 1024:>8.0f} KB"
        f"   {parquet} parquet file(s)"
    )

### What that shows

Three storage locations, two metadata stores, one session:

| Table | Metadata lives in | Bytes live in |
|---|---|---|
| `spark_catalog.default.trips` | `metastore_db/` | `spark-warehouse/trips/` |
| `spark_catalog.taxi_db.hourly_summary` | `metastore_db/` | `custom_warehouse/` |
| `taxi_cat.curated.hourly_summary` | `taxi_catalog_db/` | `taxi_catalog_db/` |

The first two share a catalog and therefore a metastore, but not a directory — that is what
`CREATE DATABASE … LOCATION` did. The third shares neither: registering a catalog replaced
the whole lookup mechanism, so `metastore_db` and `spark-warehouse` were never consulted, and
the table is not stored as Parquet at all.

That is the point worth carrying to Databricks. `metastore_db` and `spark-warehouse` are not
"how Spark stores tables" — they are the defaults of one catalog implementation. Unity Catalog
is another implementation of the same plug-in point: a different metadata service, different
storage, the same `catalog.database.table` names in your SQL.

**Optional, not marked.** `spark_catalog.default.trips` and
`taxi_cat.curated.hourly_summary` are both reachable from one session by name alone. What
would you have to know, or do, to move a table from one to the other — and which of the two
directories would each step touch?

_(replace this line)_

To clean up everything this appendix created:

```sh
rm -rf custom_warehouse taxi_catalog_db
```

Both are git-ignored, as are `spark-warehouse/` and `metastore_db/`.

In [ ]:
# This also shuts down the Spark UI.
spark.stop()